# Study Tutor — Project

**AI Agentic Engineering · Ingeniería de Sistemas · Universidad de Santander**

A conversational assistant that turns a student's own PDF (class notes or slides) into
study material: it extracts the topics, generates practice exams grounded in the source
text, and gives specific feedback on the student's answers — never a bare right/wrong.

Everything the assistant *decides* lives in this notebook. The mechanical part of the
pipeline (reading PDFs, cleaning, chunking, embedding, storing) lives in the `tutor/`
package next to it, because it is deterministic, unit-tested, and not interesting to read
cell by cell.

## Architecture

```
  data/*.pdf
      |
      v
  [ INGESTION — plain code, no LLM ]        tutor/ingest/
  pypdf -> clean -> chunk -> embed -> ChromaDB
      |
      v
  [ RETRIEVAL ]  semantic search over the chunks    tutor/vectorstore.py
      ^
      | every agent below retrieves before it speaks
      |
  [ ORCHESTRATOR ] -- routes the student's request
      |-- Topic Extractor   -> structured list of topics      (this notebook)
      |-- Exam Generator    -> questions + answer + source    (this notebook)
      +-- Grader            -> specific feedback              (this notebook)
```

## How to run this notebook

1. `pip install -r ../requirements.txt`
2. Run the setup cell below once. It creates `../.env` for you from `../.env.example`;
   open it and paste your key from https://aistudio.google.com/apikey. That file is
   git-ignored, so the key never leaves this machine — which is also why the repo ships
   the template and not the file itself.
3. Drop one or more PDFs into `../data/`.
4. **Run All.** Every cell is written to run top to bottom with no hidden state; if it
   only works out of order, that is a bug, not a quirk.

> Editing `.env` or any file in `tutor/` while the kernel is running used to have no
> effect, because Python caches imported modules and `from x import Y` copies the value.
> The setup cell now enables `autoreload` for the code and calls `config.reload()` for the
> settings, so re-running it is enough. If the printed model is not what `.env` says,
> the setup cell did not run.

> Ingestion is skipped automatically when the vector store already holds the chunks, so
> re-running the whole notebook does not burn API quota.


---
## 0. Setup


In [1]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [1]:
# Pick up edits to tutor/*.py without restarting the kernel. A Jupyter kernel
# outlives every change you make to the code it imported, and 'my fix did nothing'
# is almost always this, not the fix.
%load_ext autoreload
%autoreload 2

import logging
import sys
from pathlib import Path

# The notebook lives in notebooks/, the package one level up.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from tutor import config   # importing this creates .env from .env.example if missing

# autoreload watches .py files, not .env, so settings are re-read explicitly here.
# Every module reads config.X at call time, so this one line updates all of them.
config.reload()

# The google-genai SDK logs an automatic-function-calling notice on every
# generate_content call. We use no tools, so it is pure noise in the output.
logging.getLogger('google_genai.models').setLevel(logging.ERROR)

print('repo root      ', config.ROOT_DIR)
print('chat model     ', config.GEMINI_MODEL)
print('embed model    ', config.GEMINI_EMBED_MODEL, f'({config.EMBED_DIM} dims)')
print('chunk size     ', config.CHUNK_SIZE, 'chars, overlap', config.CHUNK_OVERLAP)
print('local fallback ', 'on' if config.ENABLE_LOCAL_FALLBACK else 'off (Gemini only)')
print('PDFs in data/  ', [p.name for p in config.DATA_DIR.glob('*.pdf')] or 'NONE - add one!')

# Fail here, loudly, rather than 20 cells later with a 404 that blames the model.
try:
    config.validate_models()
except Exception as error:
    print('\nCONFIGURATION PROBLEM:\n')
    print(error)

if config.GOOGLE_API_KEY:
    print('API key        loaded OK')
else:
    print(f'API key        MISSING -> open {config.ENV_FILE} and paste your key')


repo root       C:\Users\ACER\Desktop\AI_Agentic_Engineering_Project
chat model      gemini-3.6-flash
embed model     gemini-embedding-001 (768 dims)
chunk size      1200 chars, overlap 200
local fallback  off (Gemini only)
PDFs in data/   ['Test.pdf']
API key        loaded OK


---
## 1. Ingestion — deliberately not an agent

Reading a PDF, stripping its running headers, cutting it into chunks and embedding them
is a fully deterministic transformation. Putting an LLM in charge of it would add cost,
latency and a failure mode, and buy nothing: there is no judgement call to make. So this
stage is plain Python in `tutor/ingest/`, and the agents start only where judgement
actually begins.

Four decisions inside that package are worth knowing, because they are what make
retrieval work later:

| Decision | Why |
|---|---|
| Header/footer removal by **position + repetition** | A footer repeated on 40 pages gets embedded 40 times and floods every result list with near-identical noise. |
| Chunks respect **paragraph boundaries**, with 200-char overlap | A chunk cut mid-sentence embeds half an idea; the overlap keeps a concept that straddles two paragraphs findable from both. |
| Chunk id = **hash of its own content** | Re-ingesting the same PDF overwrites the same rows instead of duplicating them. Three test runs before the demo would otherwise triple the index. |
| `task_type` **RETRIEVAL_DOCUMENT vs RETRIEVAL_QUERY** | `gemini-embedding-001` is asymmetric: the same text embeds differently as a stored document than as a question. Using one type for both measurably hurts recall. |


In [2]:
from tutor.errors import TutorError
from tutor.ingest.pipeline import ingest
from tutor.vectorstore import get_collection

collection = get_collection()

if collection.count() > 0:
    print(f'Vector store already holds {collection.count()} chunks - skipping ingestion.')
    print('Call ingest(reset=True) if you replaced the PDFs in data/.')
else:
    try:
        stats = ingest()          # reads every PDF in data/
        for key, value in stats.items():
            print(f'{key:20} {value}')
    except TutorError as error:
        print('Ingestion could not run:\n')
        print(error)


  -> opening ChromaDB ...
  <- opening ChromaDB done in 0.2s
Vector store already holds 13 chunks - skipping ingestion.
Call ingest(reset=True) if you replaced the PDFs in data/.


### Retrieval check

Before building anything on top, confirm the retrieval layer actually finds the right
material. This is the same `semantic_search` idea from the Week 3 notebook, now backed by
a real vector database and returning provenance (file + page) alongside the score.

**Read the scores, do not just admire them.** If the top hits for a question you know is
answered in the PDF sit below ~0.4, or the text returned is off-topic, the problem is the
chunking — fix it here. Every agent downstream inherits whatever this cell returns.

The first run of this cell is the slow one: ChromaDB loads its runtime and the
Gemini client opens a TLS connection. Later runs reuse both. Every stage prints
as it starts, so a long pause with no output means something is stuck rather than
working — `python scripts/diagnose.py` times each stage separately.


In [3]:
from tutor.vectorstore import search

QUESTION = '¿Por qué podemos considerar que las vacas son animales valiosos y beneficiosos para el ser humano?'

for hit in search(QUESTION, top_k=3):
    meta = hit['metadata']
    print(f"[{hit['score']:.3f}] {meta['source']} p.{meta['page']}")
    print('   ', hit['text'][:280].replace(chr(10), ' '), '...')
    print()


  cache hit on all 1 - no API call needed
  -> searching 13 chunks ...
  <- searching 13 chunks done in 0.0s
[0.791] Test.pdf p.7
    enos agrícolas, transportar materiales y proporcionar diversos recursos. También forman parte de numerosas culturas y pueden desempeñar un papel dentro de determinados sistemas ecológicos y agrícolas. Al mismo tiempo, reconocer su importancia implica reconocer nuestra responsabil ...

[0.776] Test.pdf p.4
    relación histórica entre humanos y bovinos ha sido mucho más amplia que simplemente obtener alimentos. Sin embargo, reconocer estos beneficios no significa ignorar los problemas relacionados con la ganadería. La producción bovina puede generar impactos ambientales importantes, es ...

[0.776] Test.pdf p.3
    Las vacas y su importancia para la humanidad Durante siglos, las vacas han proporcionado recursos fundamentales para las comunidades humanas. La leche, por ejemplo, ha sido utilizada para producir una enorme variedad de alimentos, entre ellos q

---
## 2. The LLM layer — one door for every call

None of the agents below talk to Gemini directly. They all go through
`tutor.llm.generate()`, which buys three things:

| | |
|---|---|
| **One policy** | Retry, timeout and error handling are written once instead of repeated in four agents. |
| **One contract** | Structured output is schema-constrained *at decoding time*, so an agent's Pydantic model always holds. |
| **One place to swap models** | Changing provider is a line in `.env`, not a hunt through the notebook. |

### How failures are classified

The policy is not "retry everything". Three cases, three different responses:

| Failure | Response | Why |
|---|---|---|
| `429 RESOURCE_EXHAUSTED` | stop retrying at once | retrying cannot create quota |
| `503` / network | retry with backoff (2s, 4s) | Google's shared models spike under load and recover in seconds |
| anything else | **re-raise** | a bad prompt or broken schema is *our* bug and must be loud |

That last row is the one worth defending. Swallowing every error and answering
with some weaker model would replace a stack trace with a quietly worse answer.

### About the local fallback

`tutor/llm.py` can degrade to a local Qwen model through Ollama when Gemini is
unusable. It is **off** (`ENABLE_LOCAL_FALLBACK=false` in `.env`) while the agents are
being built: an 8B model on CPU is slow, and having it answer silently would mask the
very Gemini behaviour we need to see. It gets turned on before the live demo, where a
quota limit in front of the class is a real risk. `tests/test_llm.py` keeps the routing
logic honest either way, with no network calls.

> **If a cell below fails with `503 UNAVAILABLE`, nothing is broken on your side.**
> Google's shared model is busy. Re-run in a minute, or pin a specific build with
> `GEMINI_MODEL=gemini-2.5-flash` in `.env` instead of the `-latest` alias, which points
> at whichever build is under the most load.


In [4]:
from tutor import llm

print('model            ', config.GEMINI_MODEL)
print('local fallback   ', 'ENABLED' if config.ENABLE_LOCAL_FALLBACK else 'disabled (Gemini only)')
if config.ENABLE_LOCAL_FALLBACK and not llm.ollama_available():
    print('  WARNING: fallback is on but Ollama is not reachable at', config.OLLAMA_HOST)


model             gemini-3.6-flash
local fallback    disabled (Gemini only)


### Structured output, end to end

The same Pydantic model is passed to `generate()` and used to validate the reply.
Note this is not "please answer in JSON" in the prompt — the schema constrains the
model while it decodes, so it cannot emit a shape that fails validation. Every agent
from here on uses this pattern.


In [5]:
from pydantic import BaseModel, Field

from tutor.vectorstore import search


class ChunkSanityCheck(BaseModel):
    """A throwaway schema, just to prove the plumbing works before building agents."""

    language: str = Field(description='language of the text, e.g. Spanish')
    main_subject: str = Field(description='what this passage is about, in 5 words or fewer')
    is_exam_worthy: bool = Field(description='true if a question could be written from it')


sample = search('tema principal del documento', top_k=1)[0]

response = llm.generate(
    prompt=f'Analyse this passage from a student document:\n\n{sample["text"]}',
    system='You analyse study material. Answer only with the requested structure.',
    schema=ChunkSanityCheck,
)

print('answered by:', response.provider, '|', response.model)
print(response.parse(ChunkSanityCheck))


  cache hit on all 1 - no API call needed
  -> searching 13 chunks ...
  <- searching 13 chunks done in 0.0s
  -> gemini-3.6-flash ...
     still waiting on gemini-3.6-flash ... 5s
  <- gemini-3.6-flash failed after 9.8s
     Gemini busy (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp) - retry 1/4 in 2.3s
  -> gemini-3.6-flash (attempt 2/5) ...
     still waiting on gemini-3.6-flash (attempt 2/5) ... 5s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 10s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 15s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 20s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 25s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 30s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 35s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 40s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 45s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 50s
     still

### The fallback drill (inactive)


In [6]:
# Only meaningful once ENABLE_LOCAL_FALLBACK=true. A safety net nobody tested is
# not a safety net, so this drill simulates a 429 and checks the answer still arrives.
if config.ENABLE_LOCAL_FALLBACK and llm.ollama_available():
    real_gemini = llm._call_gemini

    class SimulatedRateLimit(Exception):
        code = 429

    def rate_limited(*args, **kwargs):
        raise SimulatedRateLimit('429 RESOURCE_EXHAUSTED (simulated)')

    try:
        llm._call_gemini = rate_limited
        forced = llm.generate('Reply with the single word: alive.', temperature=0.0)
        print('answered by:', forced.provider, '|', forced.model)
    finally:
        llm._call_gemini = real_gemini   # always restore, even if the cell raises
else:
    print('Local fallback disabled - skipping the drill. This is expected right now.')


Local fallback disabled - skipping the drill. This is expected right now.


---
## 3. Topic Extractor agent — *next step*

## 4. Exam Generator agent — *next step*

## 5. Grader agent — *next step*

## 6. Orchestrator + conversation memory — *next step*


---
## Appendix — secret leak check

This notebook is committed **with its outputs**, so the professor can read the results on
GitHub without running anything. That convenience has a cost: a stray `print` or an
exception traceback from `google-genai` can put the API key inside a saved output, and
`.gitignore` does not protect you there — `.env` is ignored, the notebook is not.

Run this cell after **Save**, and before every commit.


In [ ]:
import json, re

nb_path = Path.cwd() / 'tutor.ipynb'
raw = nb_path.read_text(encoding='utf-8')

problems = []
if config.GOOGLE_API_KEY and config.GOOGLE_API_KEY in raw:
    problems.append('your GOOGLE_API_KEY appears verbatim in the saved notebook')
for match in set(re.findall(r'AIza[0-9A-Za-z_\-]{20,}', raw)):
    problems.append(f'a Google API key pattern appears: {match[:8]}...')

if problems:
    print('DO NOT COMMIT:')
    for problem in problems:
        print(' -', problem)
    print('\nClear the offending cell output (Cell > Current Outputs > Clear), save, re-run this check,')
    print('and rotate the key at aistudio.google.com if it was ever pushed.')
else:
    print('Clean: no API key found in the saved notebook. Safe to commit.')
